# New API column generation

In [ ]:
"""
Created on Wednesday Jan 14 2026

@author: Laura Rueda
"""

import pandas as pd
import requests
import xml.etree.ElementTree as ET
import time
import os
from tqdm import tqdm # Library to show progress bars

In [ ]:
# -LOAD UNIQUE CODES 
# We use the mapping file we created before, or load the unique ones from the large CSV
print("Loading unique RX codes...")
df_codes = pd.read_csv('data/mappings/rx_code_mapping.csv')
unique_codes = df_codes['RX_CODE'].unique()
print(f"Total codes to check from API: {len(unique_codes)}")

In [ ]:
# DEFINE THE API FUNCTION 
def get_rxnorm_ingredient(rx_code_raw):
    """
    1. Clean the code (remove 'RXNORM:')
    2. Call the API
    3. Search for <tty>IN</tty> (Ingredient)
    4. Return the name
    """
    
    # 1. Cleaning: If it comes as "RXNORM:12345", we keep "12345"
    if isinstance(rx_code_raw, str) and ':' in rx_code_raw:
        rxcui = rx_code_raw.split(':')[-1]
    else:
        rxcui = str(rx_code_raw)
        
    # API URL (Endpoint 'allrelated')
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allrelated.xml"
    
    try:
        # 2. Call the API
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            # 3. Parse XML
            root = ET.fromstring(response.content)
            
            # We look for all concept groups
            # The structure is: allRelatedGroup -> conceptGroup -> tty
            for concept_group in root.findall(".//conceptGroup"):
                tty = concept_group.find("tty")
                
                # WE LOOK EXACTLY FOR THE TAG <tty>IN</tty> (Ingredient)
                if tty is not None and tty.text == "IN":
                    # If we find the 'IN' group, we take the name of the first concept inside
                    name_tag = concept_group.find(".//name")
                    if name_tag is not None:
                        return name_tag.text
            
            # If we reach here, it means no ingredient was found (could be a device, etc.)
            return "NO_INGREDIENT_FOUND"
            
        else:
            return "API_ERROR"
            
    except Exception as e:
        return "CONNECTION_ERROR"



In [ ]:
#  AUTOMATED LOOP WITH PROGRESS BAR 
results = {}

print("Starting API queries (This may take a few minutes)...")
# tqdm shows us a nice progress bar
for code in tqdm(unique_codes):
    ingredient_name = get_rxnorm_ingredient(code)
    results[code] = ingredient_name
    
    # IMPORTANT: Small pause to avoid overwhelming the API and getting banned
    time.sleep(0.05) 

# --- STEP 4: SAVE RESULTS ---
print("Creating results DataFrame...")
df_api_results = pd.DataFrame(list(results.items()), columns=['RX_CODE', 'RX_NAME_API'])

# We save to CSV so we don't have to run this again
os.makedirs('data/api_results', exist_ok=True)
df_api_results.to_csv('data/api_results/rx_ingredient_mapping.csv', index=False)

print("\n--- SAMPLE RESULTS ---")
print(df_api_results.head(10))
print(f"\n✅ Process completed. File saved at 'data/api_results/rx_ingredient_mapping.csv'")